In [1]:
from write_features_functions import *
from util import *
from manual_annotations_functions import *

import os
import pickle
import h5py
import numpy as np
import json
import geojson
from scipy.io import loadmat, savemat
from tqdm import tqdm

In [2]:
# Define paths and parameters
pth0 = r'\\10.162.80.16\Andre\data\Stardist\Healthy nPOD'
nm = 'nondiabetic'
dtm = '1_1_2025'
model = 'panin_healthy'
target_class = 6  # None or class where immune cells are located

pthndpi = os.path.join(pth0, nm)
pthstardist = os.path.join(pth0, nm, f'StarDist_{dtm}_{model}')
json_path = os.path.join(pthstardist, 'json')
pth_immunecells = os.path.join(pthstardist, 'immune_cell_annotations_AF')
selection_json_path = os.path.join(pth_immunecells, 'json')

pthpkl_classes = os.path.join(pthstardist, 'stardist_feature_df_pickles_classes')
os.makedirs(selection_json_path, exist_ok=True)

pthidxs = os.path.join(pth_immunecells, 'immune_idxs')
pthidxs_geojson = os.path.join(pthidxs, 'geojsons')
os.makedirs(pthidxs_geojson, exist_ok=True)

outpklpth = os.path.join(pth_immunecells, 'stardist_feature_df_pickles_classes')
os.makedirs(outpklpth, exist_ok=True)

pth_pixres = os.path.join(pthndpi, 'segmentation_analysis', 'pix_res_info')

print(pthpkl_classes)
print(pthidxs)
print(outpklpth)
print(target_class)

\\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\stardist_feature_df_pickles_classes
\\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\immune_cell_annotations_AF\immune_idxs
\\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\immune_cell_annotations_AF\stardist_feature_df_pickles_classes
6


In [3]:
# Add immune cell labels to dataframe pickles
def add_newlabel_to_df_pkl(pthpkl, cluster1_pth, outpklpth, target_class=None):
    pkl_pthlist = [os.path.join(pthpkl, f) for f in os.listdir(pthpkl) if f.endswith(".pkl")]
    pthmatfiles = os.path.join(outpklpth, 'mat')
    os.makedirs(pthmatfiles, exist_ok=True)

    print(f'saving here: {outpklpth}')
    for i, dfnm in enumerate(pkl_pthlist):
        nm = os.path.splitext(os.path.basename(dfnm))[0]
        print(f"Saving mat file {nm}  {i+1}/{len(pkl_pthlist)}")

        dst = os.path.join(outpklpth, nm + '.pkl')
        matnm = os.path.join(cluster1_pth, nm + '.mat')

        if os.path.exists(dst):
            print(f"MAT file already exists, skipping the file ID {nm}")
            continue

        with open(dfnm, 'rb') as f:
            df = pickle.load(f)

        with h5py.File(matnm, 'r') as mat_data:
            idx_list = np.array(mat_data['cluster1']).flatten()
            print("Extracted cluster1 data")

        df['cluster1'] = idx_list
        df['previous_class_name'] = df['class_name']
        max_class_name = df['class_name'].max()
        new_class_X = max_class_name + 1

        if target_class is not None:
            df.loc[(df['class_name'] == target_class) & (df['cluster1'] == 1), 'class_name'] = new_class_X
        else:
            df.loc[df['cluster1'] == 1, 'class_name'] = new_class_X

        df['new_class_name'] = df['class_name']
        df = df.drop(columns=['class_name']).rename(columns={'new_class_name': 'class_name'})

        with open(dst, 'wb') as f:
            pickle.dump(df, f)
        print(f"Saved updated DataFrame")

        new_mat_path = os.path.join(pthmatfiles, f"{nm}.mat")
        savemat(new_mat_path, {'features': df.to_numpy().astype(np.float32), 'feature_names': df.columns.tolist()})
        print(f"Saved matfile")


In [4]:
add_newlabel_to_df_pkl(pthpkl_classes, pthidxs, outpklpth, target_class)

saving here: \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\immune_cell_annotations_AF\stardist_feature_df_pickles_classes
Saving mat file 6516-001_HE  1/101
MAT file already exists, skipping the file ID 6516-001_HE
Saving mat file 6516-005_HE  2/101
MAT file already exists, skipping the file ID 6516-005_HE
Saving mat file 6516-009_HE  3/101
MAT file already exists, skipping the file ID 6516-009_HE
Saving mat file 6516-013_HE  4/101
MAT file already exists, skipping the file ID 6516-013_HE
Saving mat file 6516-017_HE  5/101
MAT file already exists, skipping the file ID 6516-017_HE
Saving mat file 6516-021_HE  6/101
MAT file already exists, skipping the file ID 6516-021_HE
Saving mat file 6516-025_HE  7/101
MAT file already exists, skipping the file ID 6516-025_HE
Saving mat file 6516-029_HE  8/101
MAT file already exists, skipping the file ID 6516-029_HE
Saving mat file 6516-033_HE  9/101
MAT file already exists, skipping the file ID 6516-03

In [5]:
# Class names and colors
class_names = ["islets of Langerhans", "pancreatic ducts", "blood vessels", "fat", "acini", "stroma", "background", "nerves", "immune cells"]
class_colors = [[70, 220, 220], [80, 70, 220], [70, 180, 80], [220, 220, 80], [130, 50, 190], [240, 180, 230], [255, 255, 255], [160, 130, 40], [0, 0, 0]]
class_colors_norm = [[c / 255 for c in color] for color in class_colors]

# Extract pixel resolution
pthmatfiles = get_sorted_files(pth_pixres, '.mat')
mat = loadmat(pthmatfiles[0])
pixelres = float(mat['pix_res'][0][0][0])
print(f"Pixel Resolution: {pixelres}")

Pixel Resolution: 0.504


In [6]:
# Generate GeoJSON contours with classes
def make_geojson_contours_with_classes_immune(out_pth_json, ds_amt=1, ds_mask=1 / 0.4416, class_names=None, class_colors=None, gj=1, pth_pkl_classes=None):
    if gj != 1:
        print("GeoJSON generation is skipped.")
        return

    if class_names is None or class_colors is None:
        raise ValueError("Both class_names and class_colors must be provided.")

    out_pth_contours = os.path.join(out_pth_json, 'geojsons', '32_polys_20x_labels_immune')
    os.makedirs(out_pth_contours, exist_ok=True)
    json_pth_list = [f for f in os.listdir(out_pth_json) if f.endswith('.json')]

    for p, file in enumerate(json_pth_list):
        nm = os.path.basename(file)
        file_name, _ = os.path.splitext(nm)
        new_fn = os.path.join(out_pth_contours, file_name + '.geojson')

        if os.path.exists(new_fn):
            print(f'Skipping {new_fn}')
            continue

        with open(os.path.join(pth_pkl_classes, file_name + '.pkl'), 'rb') as f:
            df = pickle.load(f)

        with open(os.path.join(out_pth_json, file)) as f:
            segmentation_data = json.load(f)

        data_list = get_ds_data(segmentation_data, ds_amt)
        GEOdata = []

        for j, (centroid, contour) in tqdm(enumerate(data_list), total=len(data_list), desc="Processing contours"):
            y, x = int(round(centroid[0])), int(round(centroid[1]))
            label = df.iloc[j]['class_name']

            if label > 0:
                classification_name = class_names[label - 1]
                classification_color = [int(c * 255) for c in class_colors[label - 1]]
                contour = [[coord for coord in xy[::-1]] for xy in contour]
                contour.append(contour[0])

                dict_data = {
                    "type": "Feature",
                    "id": "PathCellObject",
                    "geometry": {"type": "Polygon", "coordinates": [contour]},
                    "properties": {'objectType': 'annotation', 'classification': {'name': classification_name, 'color': classification_color}}
                }

                GEOdata.append(dict_data)

        with open(new_fn, 'w') as outfile:
            geojson.dump(GEOdata, outfile)
        print(f'Finished {new_fn}')

In [7]:
make_geojson_contours_with_classes_immune(json_path, ds_amt=1, ds_mask=1/pixelres, class_names=class_names, class_colors=class_colors_norm, pth_pkl_classes=outpklpth)

Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-001_HE.geojson
Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-005_HE.geojson
Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-009_HE.geojson
Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-013_HE.geojson
Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-017_HE.geojson
Skipping \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-021_HE.geojson
Skipping \\10.162.80.16\Andr

Processing contours: 100%|██████████| 835763/835763 [01:14<00:00, 11203.99it/s]


Finished \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-041_HE.geojson


Processing contours: 100%|██████████| 867656/867656 [01:14<00:00, 11632.00it/s]


Finished \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-045_HE.geojson


...

Finished \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-393_HE.geojson


Processing contours: 100%|██████████| 491293/491293 [00:39<00:00, 12523.58it/s]


Finished \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-397_HE.geojson


Processing contours: 100%|██████████| 551584/551584 [00:47<00:00, 11633.92it/s]


Finished \\10.162.80.16\Andre\data\Stardist\Healthy nPOD\nondiabetic\StarDist_1_1_2025_panin_healthy\json\geojsons\32_polys_20x_labels_immune\6516-401_HE.geojson
